In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from timeseries import read_timeseries_csv
from scenarios.price_scenarios import build_price_matrix

zone = "centro_sud"

price_matrix = build_price_matrix()
price_matrix.head()

price_scenario Delayed transition                                      \
weather_year                 1991        1992        1993        1994   
contract_year                                                           
2026                   104.521571  104.982755  103.990135  104.093857   
2027                   102.100541  102.561726  101.569106  101.672827   
2028                    99.679512  100.140696   99.148076   99.251798   
2029                    97.258483   97.719667   96.727047   96.830769   
2030                    94.837453   95.298638   94.306018   94.409740   

price_scenario                                                              \
weather_year          1995        1996        1997        1998        1999   
contract_year                                                                
2026            103.284847  104.649355  103.784503  103.193555  103.306062   
2027            100.863818  102.228326  101.363474  100.772526  100.885032   
2028             98.442789   99.807296   98.942445   98.351496   98.464003   
2029             96.021759   97.386267   96.521415   95.930467   96.042974   
2030             93.600730   94.965238   94.100386   93.509438   93.621944   

price_scenario              ... Net Zero 2050                          \
weather_year          2000  ...          2014        2015        2016   
contract_year               ...                                         
2026            103.668238  ...    111.006854  110.544344  109.892894   
2027            101.247209  ...    111.738950  111.276440  110.624990   
2028             98.826179  ...    112.471046  112.008536  111.357086   
2029             96.405150  ...    113.203143  112.740633  112.089183   
2030             93.984121  ...    113.935239  113.472729  112.821279   

price_scenario                                                              \
weather_year          2017        2018        2019        2020        2021   
contract_year                                                                
2026            108.738251  110.885226  109.198881  109.718072  109.699545   
2027            109.470348  111.617323  109.930977  110.450168  110.431642   
2028            110.202444  112.349419  110.663073  111.182264  111.163738   
2029            110.934540  113.081515  111.395170  111.914360  111.895834   
2030            111.666636  113.813612  112.127266  112.646457  112.627931   

price_scenario                          
weather_year          2022        2023  
contract_year                           
2026            109.787155  109.378386  
2027            110.519251  110.110482  
2028            111.251348  110.842578  
2029            111.983444  111.574675  
2030            112.715540  112.306771  

[5 rows x 99 columns]

In [2]:
solar_dir = PROJECT_ROOT / "data" / "processed" / "solar" / zone

annual_solar_cf = {}
for year in range(1991, 2024):
    cf = read_timeseries_csv(solar_dir / f"solar_cf_{year}.csv")
    annual_solar_cf[year] = cf["solar_cf"].mean()

annual_solar_cf = pd.Series(annual_solar_cf)
annual_solar_cf.describe()

count    33.000000
mean      0.200528
std       0.004835
min       0.189947
25%       0.196989
50%       0.201503
75%       0.203541
max       0.210361
dtype: float64

In [3]:
STRIKE_PRICE_SOLAR = 56.83
HOURS_PER_YEAR = 8760

pap_payment_per_mw = STRIKE_PRICE_SOLAR * annual_solar_cf * HOURS_PER_YEAR
pap_payment_per_mw.describe()

count        33.000000
mean      99829.242612
std        2406.794154
min       94561.543522
25%       98067.324857
50%      100314.622290
75%      101328.737089
max      104724.086573
dtype: float64

In [4]:
from scenarios.load_profile import load_profile_for_archetype

annual_kwh = 20_000_000
load = load_profile_for_archetype("chemicals", annual_kwh)
load.sum() / 1000

np.float64(20000.0)

In [5]:
annual_load_mwh = load.sum() / 1000
CONTRACTED_MW = 5

residual_mwh = (annual_load_mwh - annual_solar_cf * HOURS_PER_YEAR * CONTRACTED_MW).clip(lower=0)
residual_mwh.describe()

count       33.000000
mean     11216.853545
std        211.753841
min      10786.196853
25%      11084.925472
50%      11174.149015
75%      11371.870064
max      11680.314665
dtype: float64

In [6]:
payment_total = pap_payment_per_mw * CONTRACTED_MW

columns = {}
for scenario, weather_year in price_matrix.columns:
    spot_price = price_matrix[(scenario, weather_year)]
    columns[(scenario, weather_year)] = payment_total[weather_year] + residual_mwh[weather_year] * spot_price

cost_matrix = pd.DataFrame(columns)
cost_matrix.columns.names = ["price_scenario", "weather_year"]
cost_matrix.index.name = "contract_year"
cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.672498e+06  1.688981e+06  1.660627e+06  1.660836e+06   
2027                 1.645293e+06  1.661209e+06  1.633725e+06  1.633983e+06   
2028                 1.618089e+06  1.633436e+06  1.606823e+06  1.607130e+06   
2029                 1.590884e+06  1.605663e+06  1.579922e+06  1.580276e+06   
2030                 1.563680e+06  1.577891e+06  1.553020e+06  1.553423e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.657627e+06  1.686488e+06  1.657087e+06  1.648254e+06   
2027            1.630473e+06  1.658648e+06  1.630250e+06  1.621536e+06   
2028            1.603319e+06  1.630808e+06  1.603413e+06  1.594818e+06   
2029            1.576166e+06  1.602968e+06  1.576576e+06  1.568101e+06   
2030            1.549012e+06  1.575128e+06  1.549739e+06  1.541383e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.652756e+06  1.655375e+06  ...  1.756275e+06  1.736595e+06   
2027            1.625868e+06  1.628559e+06  ...  1.764649e+06  1.744772e+06   
2028            1.598981e+06  1.601744e+06  ...  1.773022e+06  1.752950e+06   
2029            1.572093e+06  1.574929e+06  ...  1.781396e+06  1.761128e+06   
2030            1.545205e+06  1.548114e+06  ...  1.789770e+06  1.769305e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.736124e+06  1.696493e+06  1.762447e+06  1.718351e+06   
2027            1.744395e+06  1.704389e+06  1.770923e+06  1.726484e+06   
2028            1.752667e+06  1.712286e+06  1.779399e+06  1.734616e+06   
2029            1.760938e+06  1.720182e+06  1.787875e+06  1.742749e+06   
2030            1.769210e+06  1.728079e+06  1.796351e+06  1.750882e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.717315e+06  1.732302e+06  1.713996e+06  1.726241e+06  
2027            1.725353e+06  1.740551e+06  1.721978e+06  1.734456e+06  
2028            1.733392e+06  1.748800e+06  1.729960e+06  1.742671e+06  
2029            1.741430e+06  1.757048e+06  1.737942e+06  1.750886e+06  
2030            1.749469e+06  1.765297e+06  1.745924e+06  1.759100e+06  

[5 rows x 99 columns]

In [7]:
def zone_annual_cf(technology, zone):
    directory = PROJECT_ROOT / "data" / "processed" / technology / zone
    by_year = {}
    for year in range(1991, 2024):
        cf = read_timeseries_csv(directory / f"{technology}_cf_{year}.csv")
        by_year[year] = cf[f"{technology}_cf"].mean()
    return pd.Series(by_year)

In [8]:
from scenarios.price_scenarios import (
    SOLAR_INSTALLED_CAPACITY_GW,
    WIND_INSTALLED_CAPACITY_GW,
    SOLAR_MERIT_ORDER_COEF,
    WIND_MERIT_ORDER_COEF,
    rescaled_price_trajectory,
    SCENARIOS,
    START_YEAR as SCENARIO_START_YEAR,
    END_YEAR as SCENARIO_END_YEAR,
    CONTRACT_START,
    CONTRACT_TERM_YEARS,
)

def zone_merit_order_adjustment(zone):
    solar_output = zone_annual_cf("solar", zone) * SOLAR_INSTALLED_CAPACITY_GW
    wind_output = zone_annual_cf("wind", zone) * WIND_INSTALLED_CAPACITY_GW

    return (SOLAR_MERIT_ORDER_COEF * (solar_output - solar_output.mean())
            + WIND_MERIT_ORDER_COEF * (wind_output - wind_output.mean()))

In [9]:
def zone_price_matrix(zone, start_year=CONTRACT_START, term_years=CONTRACT_TERM_YEARS):
    end_year = start_year + term_years - 1
    zone_adjustment = zone_merit_order_adjustment(zone)

    columns = {}
    for scenario in SCENARIOS:
        baseline = rescaled_price_trajectory(scenario, start_year, end_year)
        for weather_year in range(SCENARIO_START_YEAR, SCENARIO_END_YEAR + 1):
            columns[(scenario, weather_year)] = baseline + zone_adjustment[weather_year]

    matrix = pd.DataFrame(columns)
    matrix.columns.names = ["price_scenario", "weather_year"]
    matrix.index.name = "contract_year"
    return matrix

In [10]:
own_zone = "nord"
reference_zone = "sicilia"

price_own_zone = zone_price_matrix(own_zone)
price_reference_zone = zone_price_matrix(reference_zone)

basis_risk = price_own_zone - price_reference_zone
basis_risk.loc[CONTRACT_START].describe()

count    9.900000e+01
mean     2.153160e-15
std      1.409304e+00
min     -2.603181e+00
25%     -1.158058e+00
50%     -6.484306e-02
75%      6.184761e-01
max      3.148994e+00
Name: 2026, dtype: float64

In [11]:
STRIKE_PRICE_WIND = 72.85
CONTRACTED_MW_VPPA = 5

wind_cf_reference = zone_annual_cf("wind", reference_zone)

vppa_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_own = price_own_zone[(scenario, weather_year)]
    spot_reference = price_reference_zone[(scenario, weather_year)]
    settlement = (STRIKE_PRICE_WIND - spot_reference) * wind_cf_reference[weather_year] * HOURS_PER_YEAR * CONTRACTED_MW_VPPA
    vppa_columns[(scenario, weather_year)] = annual_load_mwh * spot_own + settlement

vppa_cost_matrix = pd.DataFrame(vppa_columns)
vppa_cost_matrix.columns.names = ["price_scenario", "weather_year"]
vppa_cost_matrix.index.name = "contract_year"
vppa_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.757212e+06  1.775726e+06  1.757814e+06  1.774660e+06   
2027                 1.732796e+06  1.751422e+06  1.734139e+06  1.750755e+06   
2028                 1.708381e+06  1.727117e+06  1.710463e+06  1.726850e+06   
2029                 1.683966e+06  1.702812e+06  1.686787e+06  1.702945e+06   
2030                 1.659551e+06  1.678507e+06  1.663112e+06  1.679040e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.732735e+06  1.717368e+06  1.732146e+06  1.734649e+06   
2027            1.710829e+06  1.700874e+06  1.707996e+06  1.711623e+06   
2028            1.688922e+06  1.684380e+06  1.683846e+06  1.688596e+06   
2029            1.667015e+06  1.667886e+06  1.659696e+06  1.665569e+06   
2030            1.645108e+06  1.651391e+06  1.635546e+06  1.642543e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.753801e+06  1.766345e+06  ...  1.841676e+06  1.819021e+06   
2027            1.731060e+06  1.743427e+06  ...  1.848311e+06  1.826407e+06   
2028            1.708318e+06  1.720509e+06  ...  1.854946e+06  1.833793e+06   
2029            1.685576e+06  1.697591e+06  ...  1.861582e+06  1.841178e+06   
2030            1.662835e+06  1.674674e+06  ...  1.868217e+06  1.848564e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.790918e+06  1.802011e+06  1.805620e+06  1.765938e+06   
2027            1.797348e+06  1.809263e+06  1.812364e+06  1.772099e+06   
2028            1.803777e+06  1.816515e+06  1.819108e+06  1.778261e+06   
2029            1.810207e+06  1.823768e+06  1.825853e+06  1.784422e+06   
2030            1.816637e+06  1.831020e+06  1.832597e+06  1.790583e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.843790e+06  1.779573e+06  1.835286e+06  1.836727e+06  
2027            1.851860e+06  1.786029e+06  1.843100e+06  1.844617e+06  
2028            1.859931e+06  1.792485e+06  1.850914e+06  1.852507e+06  
2029            1.868001e+06  1.798941e+06  1.858727e+06  1.860397e+06  
2030            1.876071e+06  1.805397e+06  1.866541e+06  1.868287e+06  

[5 rows x 99 columns]